## Model Development

**Load the engineered dataset and recreate the splits**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

features = pd.read_parquet("../outputs/features/daily_consumption_engineered.parquet")
print("Shape:", features.shape)
print("Date range:", features.index.min(), "->", features.index.max())

train = features.loc["2023-02-01":"2024-06-30"]
val = features.loc["2024-07-01":"2024-08-31"]
test = features.loc["2024-09-01":"2024-10-31"]

y_train = train["consumption_kwh"]
y_val = val["consumption_kwh"]
y_test = test["consumption_kwh"]

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")


Shape: (639, 14)
Date range: 2023-02-01 00:00:00 -> 2024-10-31 00:00:00
Train: 516, Val: 62, Test: 61


**SARIMA order selection via auto_arima**

In [2]:
import pmdarima as pm

sarima_model = pm.auto_arima(
    y_train,
    seasonal=True,
    m=7,                    # weekly seasonality
    stepwise=True,
    trace=True,
    suppress_warnings=True,
    error_action="ignore",
    max_p=5, max_q=5, max_P=2, max_Q=2,
)

print(sarima_model.summary())


Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[7] intercept   : AIC=8445.772, Time=0.63 sec
 ARIMA(0,1,0)(0,0,0)[7] intercept   : AIC=8494.695, Time=0.01 sec
 ARIMA(1,1,0)(1,0,0)[7] intercept   : AIC=8465.633, Time=0.05 sec
 ARIMA(0,1,1)(0,0,1)[7] intercept   : AIC=8450.029, Time=0.05 sec
 ARIMA(0,1,0)(0,0,0)[7]             : AIC=8492.717, Time=0.01 sec
 ARIMA(2,1,2)(0,0,1)[7] intercept   : AIC=8444.195, Time=0.37 sec
 ARIMA(2,1,2)(0,0,0)[7] intercept   : AIC=8442.485, Time=0.17 sec
 ARIMA(2,1,2)(1,0,0)[7] intercept   : AIC=8444.173, Time=0.35 sec
 ARIMA(1,1,2)(0,0,0)[7] intercept   : AIC=8442.015, Time=0.07 sec
 ARIMA(1,1,2)(1,0,0)[7] intercept   : AIC=8444.013, Time=0.12 sec
 ARIMA(1,1,2)(0,0,1)[7] intercept   : AIC=8444.013, Time=0.16 sec
 ARIMA(1,1,2)(1,0,1)[7] intercept   : AIC=8445.915, Time=0.33 sec
 ARIMA(0,1,2)(0,0,0)[7] intercept   : AIC=8440.569, Time=0.04 sec
 ARIMA(0,1,2)(1,0,0)[7] intercept   : AIC=8442.569, Time=0.08 sec
 ARIMA(0,1,2)(0,0,1)[7] intercept